In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# Stage 12 consolidation — Interpret a small improvement with paired evidence

Read `sessions/12_session.md` first. This is an executed reference, not an assessment of you.

**Evidence:** Actual saved handwriting predictions from an already-exposed image-level holdout; exploratory analysis only.

In [2]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib','inline')
from pathlib import Path
import sys, json
import numpy as np

ROOT = Path.cwd().resolve()
while not (ROOT/'data/raw/iris.csv').is_file():
    if ROOT == ROOT.parent:
        raise FileNotFoundError('Start Jupyter in the extracted listening_to_shape_lab folder.')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'src'))


## Step 1

Read the actual saved predictions. Row IDs must be unique so each image contributes one paired outcome.

Before running: write your expected result and name one way this check could fail.

In [3]:
import pandas as pd
from shape_lab.audits import paired_accuracy_audit
predictions = pd.read_csv(ROOT/'reports/stage_12/test_predictions.csv')
assert predictions.sample_id.is_unique
report = paired_accuracy_audit(predictions.true_label, predictions.pixels, predictions.combined,
                               resamples=4000, seed=2026)
print(json.dumps(report, indent=2))

{
  "sample_count": 360,
  "accuracy_a": 0.9638888888888889,
  "accuracy_b": 0.975,
  "difference_b_minus_a": 0.011111111111111112,
  "a_only_correct": 3,
  "b_only_correct": 7,
  "both_correct": 344,
  "both_wrong": 6,
  "exact_discordance_pvalue": 0.34375,
  "paired_percentile_interval": [
    -0.005555555555555556,
    0.027777777777777776
  ],
  "resamples": 4000,
  "seed": 2026,
  "confidence_level": 0.95,
  "status": "exploratory_on_exposed_predictions",
  "limitations": "Image-level independence assumed, not established; no writer IDs; no fresh holdout or selection correction. Interval conditions on fixed fitted predictions."
}


## Step 2

Derive the net improvement from the discordant counts, not from rounded percentages.

Before running: write your expected result and name one way this check could fail.

In [4]:
net = report['b_only_correct'] - report['a_only_correct']
assert net == 4
assert np.isclose(net/report['sample_count'], report['difference_b_minus_a'])
print('newly correct:', report['b_only_correct'], 'newly wrong:', report['a_only_correct'])
print('difference (percentage points):', 100*report['difference_b_minus_a'])
print('exploratory interval (percentage points):', 100*np.array(report['paired_percentile_interval']))

newly correct: 7 newly wrong: 3
difference (percentage points): 1.1111111111111112
exploratory interval (percentage points): [-0.55555556  2.77777778]


## Step 3

Save the status and assumptions with the numbers. A p-value is not a proof of either equality or superiority.

Before running: write your expected result and name one way this check could fail.

In [5]:
assert report['a_only_correct']==3 and report['b_only_correct']==7
assert report['exact_discordance_pvalue']==.34375
result = report
result['models'] = ['pixels', 'combined']
result['new_confirmatory_holdout'] = False

## Keep execution evidence separate from learning evidence

The following saves this reference calculation. Write your own explanation in `my_work/`; no learner score is assigned.

In [6]:
result['stage'] = 12
result['execution_role'] = 'reference_consolidation'
result['data_role'] = 'Actual saved handwriting predictions from an already-exposed image-level holdout; exploratory analysis only.'
output = ROOT/'reports/consolidation_v4/12.json'
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(json.dumps(result, indent=2, allow_nan=False) + '\n')
print('Saved reference result:', output.relative_to(ROOT))


Saved reference result: reports/consolidation_v4/12.json
